In [ ]:
!pip install pysam

In [ ]:
import numpy as np
import pandas as pd
import pyspark
import dxpy
import dxdata
import pysam

In [ ]:
!dx download 'Bulk/Exome sequences/Exome OQFE CRAM files/helper_files/GRCh38_full_analysis_set_plus_decoy_hla.fa'
!dx download 'Bulk/Exome sequences/Exome OQFE CRAM files/helper_files/GRCh38_full_analysis_set_plus_decoy_hla.fa.fai'
fasta = pysam.FastaFile('GRCh38_full_analysis_set_plus_decoy_hla.fa')

In [ ]:
inds_plink_df = pd.read_csv('/mnt/project/Bulk/DRAGEN WGS/DRAGEN population level WGS variants, PLINK format [500k release]/ukb24308_c1_b0_v1.psam', sep = ' ', header = None)
inds_plink_df.columns = ['FID', 'IID', 'PAT', 'MAT', 'SEX', 'REDACTED']
ind_sex = inds_plink_df.set_index('IID')['SEX'].to_dict()

In [ ]:
# Clinical data
sc = pyspark.SparkContext()
spark = pyspark.sql.SparkSession(sc)

dispensed_database_name = dxpy.find_one_data_object(classname = 'database', name = 'app*', folder = '/', name_mode = 'glob', describe = True)['describe']['name']
dispensed_dataset_id = dxpy.find_one_data_object(typename = 'Dataset', name = 'app*.dataset', folder = '/', name_mode = 'glob')['id']

dataset = dxdata.load_dataset(id = dispensed_dataset_id)
participant = dataset['participant']

field_names = ['eid', 'p34', 'p2946_i0', 'p1845_i0', 'p21003_i0']

df = participant.retrieve_fields(names = field_names, engine = dxdata.connect())

inds_sql_df = df.toPandas()
inds_sql_df['eid'] = inds_sql_df['eid'].astype(int)
inds_sql_df = inds_sql_df.dropna(subset = ['p34'])
inds_sql_df.head()

In [ ]:
ind_yob = inds_sql_df.set_index('eid')['p34'].to_dict()

In [ ]:
father_age = {}
mother_age = {}

father_incohort = {}
mother_incohort = {}

trios_df = pd.read_csv('/mnt/project/DNM/trios_pairs.txt', sep = ' ', header = None)

for _, row in trios_df.iterrows():
    
    child = int(row[0])
    father = int(row[1])
    mother = int(row[2])
    
    if child in ind_yob:
        if father in ind_yob and father in ind_sex:
            father_age[child] = ind_yob[child]-ind_yob[father]
            father_incohort[child] = True
        if mother in ind_yob and mother in ind_sex:
            mother_age[child] = ind_yob[child]-ind_yob[mother]
            mother_incohort[child] = True

In [ ]:
duo_df = pd.read_csv('/mnt/project/DNM/duo_pairs.txt', sep = ' ', header = None)

for _, row in duo_df.iterrows():
    
    child = int(row[0])
    parent = int(row[1])
    
    if child in ind_yob:
        if parent in ind_yob and parent in ind_sex:
            if child not in father_age and ind_sex[parent] == 1:
                father_age[child] = ind_yob[child]-ind_yob[parent]
                father_incohort[child] = True
            elif child not in mother_age and ind_sex[parent] == 2:
                mother_age[child] = ind_yob[child]-ind_yob[parent]
                mother_incohort[child] = True

In [ ]:
for _, row in inds_sql_df.iterrows():
    
    if row['eid'] not in father_age and pd.isna(row['p2946_i0']) == False and row['p2946_i0'] > 0 and pd.isna(row['p21003_i0']) == False:
        father_age[int(row['eid'])] = row['p2946_i0']-row['p21003_i0']
        
    if row['eid'] not in mother_age and pd.isna(row['p1845_i0']) == False and row['p1845_i0'] > 0 and pd.isna(row['p21003_i0']) == False:
        mother_age[int(row['eid'])] = row['p1845_i0']-row['p21003_i0']

# Families

In [ ]:
def get_types(dnms):

    types = {}

    for ref in ['C', 'T']:
        for alt in ['A', 'C', 'G', 'T']:
            if ref != alt:
                for prev in ['A', 'C', 'G', 'T']:
                    for next in ['A', 'C', 'G', 'T']:
                        types[f'{prev}[{ref}>{alt}]{next}'] = 0

    comp = {'A':'T', 'T':'A', 'G':'C', 'C':'G'}

    for dnm in dnms:

        chrom = dnm[0]
        pos = dnm[1]
        ref = dnm[2]
        alt = dnm[3]

        cont = fasta.fetch(f'chr{chrom}', pos-2, pos+1)
        tp = f'{cont[0]}[{ref}>{alt}]{cont[2]}'

        if ref == 'A' or ref == 'G':
            ref = comp[ref]
            alt = comp[alt]
            cont = f'{comp[cont[0]]}{comp[cont[1]]}{comp[cont[2]]}'
            cont = cont[::-1]
            tp = (comp[tp[6]]+tp[1]+comp[tp[2]]+tp[3]+ 
                   comp[tp[4]]+tp[5]+comp[tp[0]])

        types[tp] += 1

    return types

In [ ]:
fams_inds = {}
inds_fams = {}

fams_df = pd.read_csv('ukb_families.csv')

for _, row in fams_df.iterrows():
    
    inds = [int(ind) for ind in row['INDS'].split(',')]
    fam = f'{inds[0]}'
    for ind in inds[1:]:
        fam += f'_{ind}'
    
    fams_inds[fam] = inds
    for ind in inds:
        inds_fams[ind] = fam

In [ ]:
trios_dnms = {}

trios_dnms_df = pd.read_csv('/mnt/project/DNM/trios/out/trios_diffs_info.txt', sep = '\t')

for _, row in trios_dnms_df.iterrows():
    
    ind_idx = row['IID']
    ind = int(ind_idx.split('_')[0])
    
    chrom = int(row['CHROM'])
    pos = int(row['POS'])
    ref = row['ref']
    alt = row['alt']
    
    if ind not in trios_dnms:
        trios_dnms[ind] = []
    trios_dnms[ind].append((chrom, pos, ref, alt))

In [ ]:
fams_dnms = {}

for fam in fams_inds:
    fams_dnms[fam] = set()
    for ind in fams_inds[fam]:
        if ind in trios_dnms:
            for dnm in trios_dnms[ind]:
                fams_dnms[fam].add(dnm)

In [ ]:
sibs_dnms = {}

sibs_dnms_df = pd.read_csv('/mnt/project/DNM/sibs/out/sibs_diffs_info.txt', sep = '\t')

for _, row in sibs_dnms_df.iterrows():
    
    ind_idx = row['IID']
    
    chrom = int(row['CHROM'])
    pos = int(row['POS'])
    ref = row['ref']
    alt = row['alt']
    
    if ind_idx not in sibs_dnms:
        sibs_dnms[ind_idx] = []
    sibs_dnms[ind_idx].append((chrom, pos, ref, alt))

In [ ]:
sibs_pairs_df = pd.read_csv('/mnt/project/DNM/sibs_pairs.txt', sep = ' ', header = None)

for i, row in sibs_pairs_df.iterrows():
    
    sib1 = int(row[0])
    sib2 = int(row[1])
    
    b = int(i/100)
    p = i%100
            
    sib1_idx = f'{sib1}_b{b}_p{p}'
    sib2_idx = f'{sib2}_b{b}_p{p}'
        
    if sib1 in trios_dnms and sib2 in trios_dnms:
        continue
        
    if sib1_idx not in sibs_dnms and sib2_idx not in sibs_dnms:
        continue

    elif sib1_idx not in sibs_dnms:
        sibs_dnms[sib1_idx] = []
    elif sib2_idx not in sibs_dnms:
        sibs_dnms[sib2_idx] = []

    if sib1 in inds_fams and sib2 in inds_fams:
        fam = inds_fams[sib1]
        for dnm in sibs_dnms[sib1_idx]:
            fams_dnms[fam].add(dnm)
        for dnm in sibs_dnms[sib2_idx]:
            fams_dnms[fam].add(dnm)

In [ ]:
fams_types = {fam: get_types(fams_dnms[fam]) for fam in fams_dnms}

In [ ]:
rows = {}

for fam in fams_types:
    rows[fam] = fams_types[fam]

spectra_df = pd.DataFrame.from_dict(rows, orient = 'index')
spectra_df = spectra_df.reset_index().rename(columns = {'index': 'FAM'})

spectra_df.head()

In [ ]:
rates_df = pd.read_csv('ukb_fams_rates.txt', sep = ' ', header = None)
rates = dict(zip(rates_df[0], rates_df[1]))

c_a_rates_df = pd.read_csv('ukb_fams_c_a_rates.txt', sep = ' ', header = None)
c_a_rates = dict(zip(c_a_rates_df[0], c_a_rates_df[1]))

c_g_rates_df = pd.read_csv('ukb_fams_c_g_rates.txt', sep = ' ', header = None)
c_g_rates = dict(zip(c_g_rates_df[0], c_g_rates_df[1]))

c_t_rates_df = pd.read_csv('ukb_fams_c_t_rates.txt', sep = ' ', header = None)
c_t_rates = dict(zip(c_t_rates_df[0], c_t_rates_df[1]))

cpg_tpg_rates_df = pd.read_csv('ukb_fams_cpg_tpg_rates.txt', sep = ' ', header = None)
cpg_tpg_rates = dict(zip(cpg_tpg_rates_df[0], cpg_tpg_rates_df[1]))

t_a_rates_df = pd.read_csv('ukb_fams_t_a_rates.txt', sep = ' ', header = None)
t_a_rates = dict(zip(t_a_rates_df[0], t_a_rates_df[1]))

t_c_rates_df = pd.read_csv('ukb_fams_t_c_rates.txt', sep = ' ', header = None)
t_c_rates = dict(zip(t_c_rates_df[0], t_c_rates_df[1]))

t_g_rates_df = pd.read_csv('ukb_fams_t_g_rates.txt', sep = ' ', header = None)
t_g_rates = dict(zip(t_g_rates_df[0], t_g_rates_df[1]))

In [ ]:
spectra_df['rate'] = spectra_df['FAM'].map(rates)
spectra_df['c_a_rate'] = spectra_df['FAM'].map(c_a_rates)
spectra_df['c_g_rate'] = spectra_df['FAM'].map(c_g_rates)
spectra_df['c_t_rate'] = spectra_df['FAM'].map(c_t_rates)
spectra_df['cpg_tpg_rate'] = spectra_df['FAM'].map(cpg_tpg_rates)
spectra_df['t_a_rate'] = spectra_df['FAM'].map(t_a_rates)
spectra_df['t_c_rate'] = spectra_df['FAM'].map(t_c_rates)
spectra_df['t_g_rate'] = spectra_df['FAM'].map(t_g_rates)

spectra_df.head()

In [ ]:
spectra_df['source'] = np.where(spectra_df.index < 1002, 'trio', 'sib')

spectra_df.head()

In [ ]:
def parental_ages(inds, parental_age):
    
    available = [int(i) for i in inds if int(i) in parental_age]
    
    if len(available) == 0:
        return np.nan
    
    if len(available) == len(inds):
        return [parental_age[int(i)] for i in inds]
    
    ref = int(available[0])
    parent_yob = ind_yob[ref] - parental_age[ref]
    inferred = [ind_yob[int(i)]-parent_yob for i in inds]
    
    if np.mean(inferred) < 10:
        print(inds)
        return np.nan
    
    return inferred

In [ ]:
spectra_df['father_ages'] = (spectra_df.iloc[:, 0].str.split('_').apply(
    lambda inds: parental_ages(inds, father_age))
)

In [ ]:
spectra_df['mean_father_age'] = (spectra_df.iloc[:, 0].str.split('_').apply(
    lambda inds: np.mean(parental_ages(inds, father_age)))
)

In [ ]:
spectra_df['father_in_cohort'] = (spectra_df.iloc[:, 0].str.split('_').apply(
    lambda inds: any(int(i) in father_incohort for i in inds))
)

In [ ]:
spectra_df['mother_ages'] = (spectra_df.iloc[:, 0].str.split('_').apply(
    lambda inds: parental_ages(inds, mother_age))
)

In [ ]:
spectra_df['mean_mother_age'] = (spectra_df.iloc[:, 0].str.split('_').apply(
    lambda inds: np.mean(parental_ages(inds, mother_age)))
)

In [ ]:
spectra_df['mother_in_cohort'] = (spectra_df.iloc[:, 0].str.split('_').apply(
    lambda inds: any(int(i) in mother_incohort for i in inds))
)

In [ ]:
spectra_df.to_csv('ukb_96_type_dnms.csv', index = False)

# Trios

In [ ]:
def get_types(dnms):

    types = {}

    for ref in ['C', 'T']:
        for alt in ['A', 'C', 'G', 'T']:
            if ref != alt:
                for prev in ['A', 'C', 'G', 'T']:
                    for next in ['A', 'C', 'G', 'T']:
                        types[f'{prev}[{ref}>{alt}]{next}'] = 0

    comp = {'A':'T', 'T':'A', 'G':'C', 'C':'G'}

    for dnm in dnms:

        chrom = dnm[0]
        pos = dnm[1]
        ref = dnm[2]
        alt = dnm[3]

        cont = fasta.fetch(f'chr{chrom}', pos-2, pos+1)
        tp = f'{cont[0]}[{ref}>{alt}]{cont[2]}'

        if ref == 'A' or ref == 'G':
            ref = comp[ref]
            alt = comp[alt]
            cont = f'{comp[cont[0]]}{comp[cont[1]]}{comp[cont[2]]}'
            cont = cont[::-1]
            tp = (comp[tp[6]]+tp[1]+comp[tp[2]]+tp[3]+ 
                   comp[tp[4]]+tp[5]+comp[tp[0]])

        types[tp] += 1

    return types

In [ ]:
def get_types_part(types):
    
    types_part = {}

    for ref in ['C', 'T']:
        for alt in ['A', 'C', 'G', 'T']:
            if ref != alt:
                types_part[f'{ref}>{alt}'] = 0
                if ref == 'C' and alt == 'T':
                    types_part['CpG>TpG'] = 0
                        
    for tp in types:
        tp_part = tp[2:5]
        if tp_part[0] == 'C' and tp_part[2] == 'T' and tp[-1] == 'G':
            tp_part = 'CpG>TpG'
        types_part[tp_part] += types[tp]
            
    return types_part

In [ ]:
trios_spectra_df = spectra_df[spectra_df['source'] == 'trio']

In [ ]:
twins = {}

twins_pairs_df = pd.read_csv('/mnt/project/DNM/twins_pairs.txt', sep = ' ', header = None)

for _, row in twins_pairs_df.iterrows():
    twins[row[0]] = row[1]
    twins[row[1]] = row[0]

In [ ]:
trio_father_age = {}
trio_mother_age = {}

for _, row in trios_spectra_df.iterrows():
    
    father_ages = []
    mother_ages = []
    
    inds = [int(ind) for ind in row['FAM'].split('_')]
    
    for i in range(len(inds)):
        if isinstance(row['father_ages'], list):
            trio_father_age[inds[i]] = row['father_ages'][i]
            if inds[i] in twins:
                trio_father_age[twins[inds[i]]] = row['father_ages'][i]
        if isinstance(row['mother_ages'], list):
            trio_mother_age[inds[i]] = row['mother_ages'][i]
            if inds[i] in twins:
                trio_mother_age[twins[inds[i]]] = row['mother_ages'][i]
    
print(len(trio_father_age), len(trio_mother_age))

In [ ]:
trios_dnms = {}

trios_dnms_df = pd.read_csv('/mnt/project/DNM/trios/out/trios_diffs_info.txt', sep = '\t')

for _, row in trios_dnms_df.iterrows():
    
    ind_idx = row['IID']
    ind = int(ind_idx.split('_')[0])
    
    chrom = int(row['CHROM'])
    pos = int(row['POS'])
    ref = row['ref']
    alt = row['alt']
    
    if ind not in trios_dnms:
        trios_dnms[ind] = []
    trios_dnms[ind].append((chrom, pos, ref, alt))

In [ ]:
with open('ukb_trios_dnms_parental_ages.tsv', 'w') as f:
    f.write('ind\tind_dnms\tind_father_age\tind_mother_age\n')
    for ind in trios_dnms:
        f.write(f'{ind}\t{get_types_part(get_types(trios_dnms[ind]))}\t{trio_father_age[ind]}\t{trio_mother_age[ind]}\n')

# Sibs

In [ ]:
sibs_spectra_df = spectra_df[spectra_df['FAM'].str.split('_').str.len() > 1]

In [ ]:
fams = 0

sib_father_age = {}
sib_mother_age = {}

sib_father_incohort = {}
sib_mother_incohort = {}

for _, row in sibs_spectra_df.iterrows():
    
    if (isinstance(row['father_ages'], list) or 
        isinstance(row['mother_ages'], list)):
        fams += 1
    
    father_ages = []
    mother_ages = []
    
    inds = [int(ind) for ind in row['FAM'].split('_')]
    
    for i in range(len(inds)):
        if isinstance(row['father_ages'], list):
            sib_father_age[inds[i]] = row['father_ages'][i]
            sib_father_incohort[inds[i]] = row['father_in_cohort']
            if inds[i] in twins:
                sib_father_age[twins[inds[i]]] = row['father_ages'][i]
                sib_father_incohort[twins[inds[i]]] = row['father_in_cohort']
        if isinstance(row['mother_ages'], list):
            sib_mother_age[inds[i]] = row['mother_ages'][i]
            sib_mother_incohort[inds[i]] = row['mother_in_cohort']
            if inds[i] in twins:
                sib_mother_age[twins[inds[i]]] = row['mother_ages'][i]
                sib_mother_incohort[twins[inds[i]]] = row['mother_in_cohort']
    
print(len(sib_father_age), len(sib_mother_age), fams)

In [ ]:
sibs_dnms = {}

sibs_dnms_df = pd.read_csv('/mnt/project/DNM/sibs/out/sibs_diffs_info.txt', sep = '\t')

for _, row in sibs_dnms_df.iterrows():
    
    ind_idx = row['IID']
    
    chrom = int(row['CHROM'])
    pos = int(row['POS'])
    ref = row['ref']
    alt = row['alt']
    
    if ind_idx not in sibs_dnms:
        sibs_dnms[ind_idx] = []
    sibs_dnms[ind_idx].append((chrom, pos, ref, alt))

In [ ]:
sibs_pairs = {}
sibs_pairs_dnms = {}

sibs_pairs_df = pd.read_csv('/mnt/project/DNM/sibs_pairs.txt', sep = ' ', header = None)

for i, row in sibs_pairs_df.iterrows():
    
    sib1 = int(row[0])
    sib2 = int(row[1])
    
    b = int(i/100)
    p = i%100
            
    sib1_idx = f'{sib1}_b{b}_p{p}'
    sib2_idx = f'{sib2}_b{b}_p{p}'
        
    if sib1_idx in sibs_dnms or sib2_idx in sibs_dnms:
        sibs_pairs[(sib1, sib2)] = True
        sibs_pairs_dnms[(sib1, sib2)] = (get_types_part(get_types(sibs_dnms[sib1_idx] if sib1_idx in sibs_dnms else [])), 
                                         get_types_part(get_types(sibs_dnms[sib2_idx] if sib2_idx in sibs_dnms else [])))

In [ ]:
cnt_father_ages = 0
cnt_mother_ages = 0

cnt_father_incohort = 0
cnt_mother_incohort = 0

with open('ukb_sibs_dnms_parental_ages.tsv', 'w') as f:
    
    f.write('sib1\tsib2\tsib1_dnms\tsib2_dnms\tsib1_father_age\tsib2_father_age\tsib1_mother_age\tsib2_mother_age\n')
    
    for sib1, sib2 in sibs_pairs:
        
        sib1_father_age = 'NA'
        sib2_father_age = 'NA'
        sib1_mother_age = 'NA'
        sib2_mother_age = 'NA'
        
        if sib1 in sib_father_age:
            
            sib1_father_age = sib_father_age[sib1]
            sib2_father_age = sib_father_age[sib2]
            cnt_father_ages += 2
            
            if sib_father_incohort[sib1] == True:
                cnt_father_incohort += 2
            
        if sib1 in sib_mother_age:
            
            sib1_mother_age = sib_mother_age[sib1]
            sib2_mother_age = sib_mother_age[sib2]
            cnt_mother_ages += 2
            
            if sib_mother_incohort[sib1] == True:
                cnt_mother_incohort += 2
            
        f.write(f'{sib1}\t{sib2}\t{sibs_pairs_dnms[(sib1, sib2)][0]}\t{sibs_pairs_dnms[(sib1, sib2)][1]}\t{sib1_father_age}\t{sib2_father_age}\t{sib1_mother_age}\t{sib2_mother_age}\n')
        
print(cnt_father_ages, cnt_mother_ages, 
      cnt_father_incohort, cnt_mother_incohort,
      round((cnt_father_incohort+cnt_mother_incohort)/(cnt_father_ages+cnt_mother_ages), 4))